### Notebook 10 : End-to-End Integration Test

##### 1. Notebook Purpose

- Notebook 10 validates the complete multi-agent customer-support workflow end to end.

- Unlike the unit tests in the previous notebooks, these tests use the real workflow components:

    - Coordinator Agent
    - SQL Agent
    - Prediction Agent
    - Vector Search Agent
    - Retention Agent
    - Final Response Agent
    - Multi-Agent Orchestrator
    - Databricks SQL / Spark data
    - Churn model-serving endpoint
    - Databricks Vector Search
    - Databricks LLM endpoint

- The purpose is to verify that all previously developed components work correctly together through one shared MultiAgentState.


##### 2. Technologies Used


- Python
- Databricks
- Pydantic
- TypedDict shared state
- Databricks Model Serving
- Databricks Vector Search
- Large Language Models
- Dependency-based multi-agent orchestration
- End-to-end integration testing


##### 3. Architecture

``` text

User Request
     │
     ▼
Coordinator Agent
     │
     ▼
Execution Plan
     │
     ▼
Multi-Agent Orchestrator
     │
     ├──────── SQL Agent
     │
     ├──────── Prediction Agent
     │
     ├──────── Vector Search Agent
     │
     ├──────── Retention Agent
     │
     └──────── Final Response Agent
                   │
                   ▼
          Final Grounded Response

```

##### 4. Load the Locked Notebooks

In [0]:
%run ./09_end_to_end_multi_agent_orchestration

##### 5. Imports

In [0]:
from typing import List

##### 6. End-to-End Test Helper

In [0]:
def run_end_to_end_test(
    test_name: str,
    user_request: str,
    expected_agents: List[str],
    coordinator_llm_invoke: LLMInvokeFunction,
    final_response_llm_invoke: LLMInvokeFunction,
    sql_analytics_tool: SQLAnalyticsFunction,
    prediction_tool: PredictionToolFunction,
    vector_search_tool: VectorSearchToolFunction,
    retention_tool: RetentionToolFunction,
    unexpected_agents: Optional[List[str]] = None,
) -> MultiAgentState:
    """
    Run one real end-to-end multi-agent workflow
    and perform common workflow validations.
    """

    print("=" * 100)
    print(test_name)
    print("=" * 100)

    print("\nUSER REQUEST:")
    print(user_request)

    final_state = run_multi_agent_workflow(
    user_request=user_request,
    coordinator_llm_invoke=coordinator_llm_invoke,
    final_response_llm_invoke=final_response_llm_invoke,
    sql_analytics_tool=sql_analytics_tool,
    prediction_tool=prediction_tool,
    vector_search_tool=vector_search_tool,
    retention_tool=retention_tool,
    )

    # =========================================================
    # Coordinator
    # =========================================================

    coordinator_result = final_state.get(
        "coordinator_result"
    )

    assert coordinator_result is not None
    assert coordinator_result.status == "success"
    assert coordinator_result.execution_plan

    # =========================================================
    # Workflow errors
    # =========================================================

    assert final_state["errors"] == []

    # =========================================================
    # Agent results
    # =========================================================

    agent_results = final_state[
        "agent_results"
    ]

    # Expected agents must exist and succeed.
    for agent_name in expected_agents:

        assert (
            agent_name in agent_results
        ), (
            f"Expected agent result missing: "
            f"{agent_name}"
        )

        assert (
            agent_results[
                agent_name
            ].status
            == "success"
        )

    # Unexpected agents must not have executed.
    if unexpected_agents:

        for agent_name in unexpected_agents:

            assert (
                agent_name not in agent_results
            ), (
                f"Unexpected agent executed: "
                f"{agent_name}"
            )

    # =========================================================
    # Final Response
    # =========================================================

    assert (
        FINAL_RESPONSE_AGENT_NAME
        in agent_results
    )

    final_result = agent_results[
        FINAL_RESPONSE_AGENT_NAME
    ]

    assert isinstance(
        final_result,
        FinalResponseAgentResult,
    )

    assert (
        final_result.status
        == "success"
    )

    assert isinstance(
        final_result.final_response,
        str,
    )

    assert final_result.final_response.strip()

    # =========================================================
    # Workflow completion
    # =========================================================

    assert is_workflow_complete(
        state=final_state
    )

    assert (
        get_incomplete_agent_names(
            state=final_state
        )
        == []
    )

    # =========================================================
    # Display results
    # =========================================================

    print("\nEXECUTION PLAN:")

    for task in (
        coordinator_result.execution_plan
    ):
        print(
            f"- {task.agent_name} | "
            f"depends_on={task.depends_on}"
        )

    print("\nEXECUTION HISTORY:")

    for record in final_state[
        "execution_history"
    ]:
        print(record)

    print("\nAGENT RESULTS:")

    for agent_name, result in (
        agent_results.items()
    ):
        print(
            f"\n{agent_name}:"
        )
        print(result)

    print("\nFINAL RESPONSE:")
    print(
        final_result.final_response
    )

    print("\nWORKFLOW STATUS: PASS")
    print()

    return final_state

##### 7. Test 1 — SQL Analytics Workflow

In [0]:
def test_end_to_end_sql() -> None:
    """
    Verify:
        User
          ↓
        Coordinator
          ↓
        SQL Agent
          ↓
        Final Response Agent
    """

    final_state = run_end_to_end_test(
        test_name=(
            "TEST 1: SQL Analytics Workflow"
        ),
        user_request=(
            "How many customers churned?"
        ),
        expected_agents=[
            SQL_AGENT_NAME,
            FINAL_RESPONSE_AGENT_NAME,
        ],
        coordinator_llm_invoke=invoke_coordinator_llm,
        final_response_llm_invoke=invoke_final_response_llm,
        sql_analytics_tool=sql_analytics_tool,
        prediction_tool=prediction_tool,
        vector_search_tool=vector_search_tool,
        retention_tool=retention_tool,
    )

    sql_result = final_state[
        "agent_results"
    ][SQL_AGENT_NAME]

    assert (
        sql_result.sql_action
        == "count_churned_customers"
    )

    assert sql_result.sql_result

    assert (
        sql_result.sql_result[0][
            "churned_customers"
        ]
        == 1869
    )

    print(
        "SQL end-to-end test passed."
    )

##### 8. Test 2 — Prediction Workflow

In [0]:
def test_end_to_end_prediction() -> None:
    """
    Verify:
        User
          ↓
        Coordinator
          ↓
        Prediction Agent
          ↓
        Churn model endpoint
          ↓
        Final Response Agent
    """

    final_state = run_end_to_end_test(
        test_name=(
            "TEST 2: Prediction Workflow"
        ),
        user_request=(
            "Will customer 7590-VHVEG churn?"
        ),
        expected_agents=[
            PREDICTION_AGENT_NAME,
            FINAL_RESPONSE_AGENT_NAME,
        ],
        coordinator_llm_invoke=invoke_coordinator_llm,
        final_response_llm_invoke=invoke_final_response_llm,
        sql_analytics_tool=sql_analytics_tool,
        prediction_tool=prediction_tool,
        vector_search_tool=vector_search_tool,
        retention_tool=retention_tool,
    )

    prediction_result = final_state[
        "agent_results"
    ][PREDICTION_AGENT_NAME]

    assert (
        prediction_result.customer_id
        == "7590-VHVEG"
    )

    assert (
        prediction_result.predicted_category
        in {
            "Churn",
            "No Churn",
        }
    )

    # Confidence is intentionally NOT required.
    #
    # Your real model-serving endpoint currently
    # returns:
    #
    # {"predictions": [1]}
    #
    # so confidence may correctly be None.

    print(
        "Prediction end-to-end test passed."
    )

##### 9. Test 3 — Vector Search Workflow

In [0]:
def test_end_to_end_vector_search() -> None:
    """
    Verify:
        User
          ↓
        Coordinator
          ↓
        Vector Search Agent
          ↓
        Embedding model
          ↓
        Vector Search Index
          ↓
        Final Response Agent
    """

    final_state = run_end_to_end_test(
        test_name=(
            "TEST 3: Vector Search Workflow"
        ),
        user_request=(
            "Why are customers likely to "
            "cancel service?"
        ),
        expected_agents=[
            VECTOR_SEARCH_AGENT_NAME,
            FINAL_RESPONSE_AGENT_NAME,
        ],

        coordinator_llm_invoke=invoke_coordinator_llm,
        final_response_llm_invoke=invoke_final_response_llm,
        sql_analytics_tool=sql_analytics_tool,
        prediction_tool=prediction_tool,
        vector_search_tool=vector_search_tool,
        retention_tool=retention_tool,
    )

    vector_result = final_state[
        "agent_results"
    ][VECTOR_SEARCH_AGENT_NAME]

    assert isinstance(
        vector_result,
        VectorSearchAgentResult,
    )

    assert (
        vector_result.status
        == "success"
    )

    assert isinstance(
        vector_result.results,
        list,
    )

    if vector_result.results:

        for search_item in (
            vector_result.results
        ):
            assert search_item.customer_id
            assert search_item.note

    print(
        "Vector Search end-to-end test passed."
    )

##### 10. Test 4 — Full Retention Workflow

In [0]:
def test_end_to_end_retention() -> None:
    """
    Verify the complete dependency-based workflow:

        Prediction Agent
              +
        Vector Search Agent
              ↓
        Retention Agent
              ↓
        Final Response Agent
    """

    final_state = run_end_to_end_test(
        test_name=(
            "TEST 4: Full Retention Workflow"
        ),
        user_request=(
            "For customer 7590-VHVEG, "
            "predict churn risk, review similar "
            "customer notes, recommend a retention "
            "action, and provide a final response."
        ),
        expected_agents=[
            PREDICTION_AGENT_NAME,
            VECTOR_SEARCH_AGENT_NAME,
            RETENTION_AGENT_NAME,
            FINAL_RESPONSE_AGENT_NAME,
        ],

        coordinator_llm_invoke=invoke_coordinator_llm,
        final_response_llm_invoke=invoke_final_response_llm,
        sql_analytics_tool=sql_analytics_tool,
        prediction_tool=prediction_tool,
        vector_search_tool=vector_search_tool,
        retention_tool=retention_tool,
    )

    agent_results = final_state[
        "agent_results"
    ]

    prediction_result = agent_results[
        PREDICTION_AGENT_NAME
    ]

    vector_result = agent_results[
        VECTOR_SEARCH_AGENT_NAME
    ]

    retention_result = agent_results[
        RETENTION_AGENT_NAME
    ]

    final_result = agent_results[
        FINAL_RESPONSE_AGENT_NAME
    ]

    # =========================================================
    # Prediction
    # =========================================================

    assert (
        prediction_result.customer_id
        == "7590-VHVEG"
    )

    assert (
        prediction_result.predicted_category
        in {
            "Churn",
            "No Churn",
        }
    )

    # =========================================================
    # Vector Search
    # =========================================================

    assert (
        vector_result.status
        == "success"
    )

    # =========================================================
    # Retention
    # =========================================================

    assert (
        retention_result.customer_id
        == "7590-VHVEG"
    )

    assert (
        retention_result.recommended_action
        in {
            "offer_discount",
            "offer_support_package",
            "service_quality_review",
            "billing_review",
            "no_action",
        }
    )

    assert (
        retention_result.prediction_label
        == prediction_result.predicted_category
    )

    # =========================================================
    # Final Response
    # =========================================================

    assert final_result.final_response

    # =========================================================
    # Dependency execution order
    # =========================================================

    execution_history = final_state[
        "execution_history"
    ]

    successful_agent_order = [
        record.agent_name
        for record in execution_history
        if record.status == "success"
    ]

    prediction_index = (
        successful_agent_order.index(
            PREDICTION_AGENT_NAME
        )
    )

    vector_index = (
        successful_agent_order.index(
            VECTOR_SEARCH_AGENT_NAME
        )
    )

    retention_index = (
        successful_agent_order.index(
            RETENTION_AGENT_NAME
        )
    )

    final_response_index = (
        successful_agent_order.index(
            FINAL_RESPONSE_AGENT_NAME
        )
    )

    # Prediction and Vector Search may run in either
    # order during the same ready-task iteration.
    #
    # Both must finish before Retention.

    assert (
        prediction_index
        < retention_index
    )

    assert (
        vector_index
        < retention_index
    )

    # Retention must finish before Final Response.

    assert (
        retention_index
        < final_response_index
    )

    print(
        "Full Retention end-to-end test passed."
    )

##### 11. Run All End-to-End Tests

In [0]:
def test_multi_agent_end_to_end() -> None:
    """
    Run all real end-to-end workflow tests.
    """

    print("=" * 100)
    print("MULTI-AGENT END-TO-END TESTING")
    print("=" * 100)
    print()

    test_end_to_end_sql()

    test_end_to_end_prediction()

    test_end_to_end_vector_search()

    test_end_to_end_retention()

    print("=" * 100)
    print(
        "ALL MULTI-AGENT END-TO-END "
        "TESTS PASSED"
    )
    print("=" * 100)

##### 12. Optional Churn-Rate Test

In [0]:
def test_end_to_end_churn_rate() -> None:
    """
    Verify:
        User
          ↓
        Coordinator
          ↓
        SQL Agent
          ↓
        Final Response Agent
    """

    final_state = run_end_to_end_test(
        test_name=(
            "OPTIONAL TEST: Churn Rate"
        ),
        user_request=(
            "What is the churn rate?"
        ),
        expected_agents=[
            SQL_AGENT_NAME,
            FINAL_RESPONSE_AGENT_NAME,
        ],
        unexpected_agents=[
            PREDICTION_AGENT_NAME,
            VECTOR_SEARCH_AGENT_NAME,
            RETENTION_AGENT_NAME,
        ],
        coordinator_llm_invoke=(
            invoke_coordinator_llm
        ),
        final_response_llm_invoke=(
            invoke_final_response_llm
        ),
        sql_analytics_tool=(
            sql_analytics_tool
        ),
        prediction_tool=(
            prediction_tool
        ),
        vector_search_tool=(
            vector_search_tool
        ),
        retention_tool=(
            retention_tool
        ),
    )

    sql_result = final_state[
        "agent_results"
    ][SQL_AGENT_NAME]

    assert isinstance(
        sql_result,
        SQLAgentResult,
    )

    assert (
        sql_result.status
        == "success"
    )

    assert (
        sql_result.sql_action
        == "churn_rate"
    )

    assert sql_result.sql_result

    assert (
        float(
            sql_result.sql_result[0][
                "churn_rate_percent"
            ]
        )
        == 26.54
    )

    final_result = final_state[
        "agent_results"
    ][FINAL_RESPONSE_AGENT_NAME]

    assert (
        final_result.status
        == "success"
    )

    assert (
        final_result.final_response
    )

    print(
        "Churn-rate end-to-end test passed."
    )

##### 13. Key Learnings

- Multi-Agent Architecture — Learned how to decompose a complex workflow into specialized agents with clearly defined responsibilities.

- Coordinator Routing — Learned how a Coordinator Agent interprets a user request, selects only the required specialist agents, creates an execution plan, and defines task dependencies.

- Dependency Injection — Learned how LLM functions and tools can be passed into agents and the orchestrator rather than being tightly coupled to individual implementations.

- Shared State Management — Learned how agents communicate through a common MultiAgentState, including coordinator results, agent results, execution history, and errors.

- Pydantic Validation — Used typed Pydantic models to validate agent inputs, outputs, execution plans, statuses, and tool responses.

- Tool Integration and Validation — Integrated SQL analytics, model prediction, Vector Search, and retention tools while validating each tool's response before using it downstream.

- Dependency-Based Orchestration — Learned how depends_on controls execution order and prevents an agent from running until its required upstream agents have completed successfully.

- Grounded Final Responses — Ensured that the Final Response Agent generates its answer from validated successful agent results rather than independently inventing information.

- Error Handling and Debugging — Debugged namespace issues, %run behavior, missing functions, malformed/truncated LLM JSON, customer-ID extraction, incorrect dependencies, tool contracts, and token limits.

- Unit and Integration Testing — Used unit tests to validate individual orchestration behaviors and end-to-end integration tests to verify the complete workflow with real LLMs and tools.


##### 14. Conclusion


- This project implemented an end-to-end multi-agent AI system for telecom customer support using Databricks.

- Defined the multi-agent architecture and established clear responsibilities for the Coordinator, SQL, Prediction, Vector Search, Retention, and Final Response Agents. We created validated input and output schemas using Pydantic, introduced a shared workflow state, and established consistent status, execution-history, dependency, and error-handling conventions.

- The Coordinator dynamically creates execution plans based on the user's request, while the orchestrator executes only the required agents according to their dependencies. Each specialist agent integrates with its corresponding tool, validates the tool response, and stores its result in shared state. The Final Response Agent then produces a grounded response using the validated results of the completed agents.

- Finally, unit and end-to-end integration testing verified agent routing, dependency ordering, tool integration, workflow completion, error handling, and final response generation.

- This project provided practical experience in designing, implementing, debugging, and testing a modular multi-agent AI system with Databricks.


The project demonstrates how independent AI agents, deterministic tools, shared state, validation, and dependency-based orchestration can work together to build a reliable multi-agent AI application.